In [1]:
import transformers
print(transformers.__version__)

d:\mlpipeline\NLP_Projects\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]

4.40.2


In [8]:
import os
%pwd

'd:\\mlpipeline\\NLP_Projects\\research'

In [ ]:
#os.chdir("NLP_PROJECTS")

In [9]:
os.chdir("../")

In [10]:
%pwd

'd:\\mlpipeline\\NLP_Projects'

In [11]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path                  #frm config.yaml
    data_path: Path                 #frm config.yaml
    model_ckpt: Path                #frm config.yaml
    num_train_epochs: int           #frm params.yaml
    warmup_steps: int               #frm params.yaml
    per_device_train_batch_size: int #frm params.yaml
    weight_decay: float              #frm params.yaml
    logging_steps: int               #frm params.yaml
    evaluation_strategy: str         #frm params.yaml
    eval_steps: int                  #frm params.yaml
    save_steps: float                 #frm params.yaml
    gradient_accumulation_steps: int  #frm params.yaml

In [12]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories

In [13]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config=self.config.model_trainer
        params=self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config=ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            weight_decay = params.weight_decay,
            logging_steps = params.logging_steps,
            evaluation_strategy = params.evaluation_strategy,
            eval_steps = params.evaluation_strategy,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps
        )
        return model_trainer_config



In [14]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
import torch
from datasets import load_from_disk

In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        #loading the data
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir, num_train_epochs=1, warmup_steps=500,
            per_device_train_batch_size=1, per_device_eval_batch_size=1,
            weight_decay=0.01, logging_steps=10,
            evaluation_strategy='steps', eval_steps=500, save_steps=1e6, #eval for latest ver transformer 
            gradient_accumulation_steps=16
        ) 
        trainer = Trainer(model=model_pegasus, args=trainer_args,
                   tokenizer=tokenizer, data_collator=seq2seq_data_collator,
                  train_dataset=dataset_samsum_pt["test"],
                  eval_dataset=dataset_samsum_pt["validation"])  # tokenizer=tokenizer (4.40.2), no need latest ver
        
        trainer.train()

        ## Save model
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir,"pegasus-samsum-model"))
        ## Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir,"tokenizer"))




In [34]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 5.1.0
Uninstalling transformers-5.1.0:
  Successfully uninstalled transformers-5.1.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
  Using cached transformers-5.1.0-py3-none-any.whl.metadata (31 kB)
  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.1.0-py3-none-any.whl (10.3 MB)
Using cached accelerate-1.12.0-py3-none-any.whl (380 kB)

   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   -------------------- ------------------- 1/2 [transformers]
   ----

In [ ]:
#pip install accelerate>=1.1.0

Note: you may need to restart the kernel to use updated packages.


In [19]:

config = ConfigurationManager()
model_trainer_config = config.get_model_trainer_config()
model_trainer = ModelTrainer(config=model_trainer_config)
model_trainer.train()

[2026-02-13 23:26:16,537: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-02-13 23:26:16,558: INFO: common: yaml file: params.yaml loaded successfully]
[2026-02-13 23:26:16,560: INFO: common: created directory at: artifacts]
[2026-02-13 23:26:16,561: INFO: common: created directory at: artifacts/model_trainer]


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  0%|          | 0/51 [00:00<?, ?it/s]d:\mlpipeline\NLP_Projects\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
 20%|█▉        | 10/51 [14:56<1:09:00, 101.00s/it]

{'loss': 11.8486, 'grad_norm': 55.20011901855469, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.2}


 39%|███▉      | 20/51 [31:57<50:17, 97.34s/it]   

{'loss': 11.7465, 'grad_norm': 35.96889114379883, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.39}


...

 98%|█████████▊| 50/51 [1:21:33<01:33, 93.58s/it] 

{'loss': 11.4032, 'grad_norm': 41.282188415527344, 'learning_rate': 5e-06, 'epoch': 0.98}


100%|██████████| 51/51 [1:23:18<00:00, 98.01s/it]


{'train_runtime': 4998.3451, 'train_samples_per_second': 0.164, 'train_steps_per_second': 0.01, 'train_loss': 11.638534975986854, 'epoch': 1.0}


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 128, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8, 'forced_eos_token_id': 1}
